# 4a — Finding Correspondences with Nearest Neighbours

So far every experiment has quietly assumed that point $i$ in the new scan
is the same physical point as point $i$ in the reference scan. Real scans
don't work like that: after the robot moves, the laser hits the walls in
different places and there is no built-in pairing. Before we can compute a
cost we must *guess* the pairs. The standard guess is the simplest one —
**each point corresponds to the closest point in the other scan**.

**Learning goals**

1. Generate two scans of the same room with no point-to-point correspondence.
2. Find nearest neighbours by brute force, and see why it gets expensive.
3. Do the same with `scipy.spatial.KDTree`.
4. Visualise matched pairs, and see how the alignment guess affects them.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

## Two scans, no correspondence

The ray-casting LiDAR simulator from [1a](1a_polar_to_cartesian.ipynb) is
repeated below. We scan the room once from the origin and once after the
robot has moved a little and turned. Because the rays hit the walls at
different places, the two scans have no natural pairing even though they
describe the same room.

In [ ]:
def cast_ray(origin, angle, segments, max_range):
    """Distance along a ray from `origin` at `angle` to the nearest wall segment.
    Returns `max_range` if nothing is hit."""
    d = np.array([np.cos(angle), np.sin(angle)])
    best = max_range
    for p, q in segments:
        p, q = np.asarray(p, float), np.asarray(q, float)
        e = q - p
        denom = d[0] * e[1] - d[1] * e[0]
        if abs(denom) < 1e-12:          # ray parallel to this wall
            continue
        w = p - origin
        t = (w[0] * e[1] - w[1] * e[0]) / denom   # distance along the ray
        u = (w[0] * d[1] - w[1] * d[0]) / denom   # position along the segment (0..1)
        if t >= 0 and 0 <= u <= 1 and t < best:
            best = t
    return best


def lidar_scan(pose, segments, num_rays=360, max_range=10.0):
    """Simulate a 2D LiDAR at `pose = (x, y, theta)`.
    Returns (angles, ranges) in the *robot* frame; angle 0 is the robot's +x axis."""
    x, y, theta = pose
    angles = np.linspace(-np.pi, np.pi, num_rays, endpoint=False)
    ranges = np.array([cast_ray(np.array([x, y]), theta + a, segments, max_range) for a in angles])
    return angles, ranges


def polar_to_cartesian(angles, ranges):
    """Convert (angle, range) pairs to an (N, 2) array of (x, y) points."""
    return np.column_stack((ranges * np.cos(angles), ranges * np.sin(angles)))

In [ ]:
# A simple room: a 10 m x 8 m rectangle with a pillar in one corner.
room = [
    ((-5, -4), (5, -4)),   # bottom wall
    ((5, -4), (5, 4)),     # right wall
    ((5, 4), (-5, 4)),     # top wall
    ((-5, 4), (-5, -4)),   # left wall
    ((2, 1), (3, 1)), ((3, 1), (3, 2)), ((3, 2), (2, 2)), ((2, 2), (2, 1)),  # pillar
]

def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def to_world(points, pose):
    """Points in the robot frame -> world frame, for a robot at `pose`."""
    x, y, theta = pose
    return points @ rotation_matrix(theta).T + np.array([x, y])

pose_0 = (0.0, 0.0, 0.0)
pose_1 = (0.4, 0.2, np.radians(5))       # the robot's true motion

ref = polar_to_cartesian(*lidar_scan(pose_0, room, num_rays=240))
new = polar_to_cartesian(*lidar_scan(pose_1, room, num_rays=240))

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(ref[:, 0], ref[:, 1], s=10, label="reference scan (robot at origin)")
ax.scatter(new[:, 0], new[:, 1], s=10, label="new scan (robot moved, plotted as if it hadn't)")
ax.axis("equal"); ax.legend(); ax.grid(True)
plt.show()

## Brute-force nearest neighbours

For every point in the new scan, compute its distance to *every* point in
the reference and take the smallest. With NumPy broadcasting this is three
lines — but it builds an $N \times M$ distance matrix, so the cost grows
with the *product* of the two scan sizes.

In [ ]:
def nearest_brute_force(query, reference):
    """For each query point, index of and distance to the closest reference point."""
    diff = query[:, None, :] - reference[None, :, :]        # (N, M, 2)
    dist = np.sqrt((diff ** 2).sum(axis=2))                  # (N, M)
    idx = dist.argmin(axis=1)
    return idx, dist[np.arange(len(query)), idx]

idx_bf, dist_bf = nearest_brute_force(new, ref)
print("first 5 matches (new -> ref):", idx_bf[:5])
print("mean match distance: %.3f m" % dist_bf.mean())

## The same thing with a KD-tree

A **k-d tree** sorts the reference points into a hierarchy of boxes so that
a query only has to look inside nearby boxes. Building the tree is a
one-off cost; each query then takes roughly $\log M$ work instead of $M$.
SciPy provides one ready-made.

In [ ]:
tree = KDTree(ref)
dist_kd, idx_kd = tree.query(new)          # note: SciPy returns (distances, indices)

print("same matches as brute force:", np.array_equal(idx_bf, idx_kd))

### How much faster?

Random point sets of increasing size make the scaling obvious.

In [ ]:
sizes = [200, 1000, 3000, 6000]
t_brute, t_tree = [], []
rng = np.random.default_rng(0)
for n in sizes:
    a = rng.uniform(-10, 10, size=(n, 2))
    b = rng.uniform(-10, 10, size=(n, 2))

    t0 = time.perf_counter(); nearest_brute_force(a, b); t_brute.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); KDTree(b).query(a);        t_tree.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sizes, t_brute, "o-", label="brute force (N×M matrix)")
ax.plot(sizes, t_tree, "s-", label="KDTree build + query")
ax.set_xlabel("points per scan"); ax.set_ylabel("seconds"); ax.set_yscale("log")
ax.grid(True, which="both"); ax.legend()
plt.show()

## Visualising the matches

Draw a line from each new-scan point to the reference point it was paired
with. On the left the new scan is left where it is (as if the robot hadn't
moved); on the right it is first moved by the *true* motion.

In [ ]:
def draw_matches(ax, query, reference, title, max_lines=None):
    dist, idx = KDTree(reference).query(query)
    ax.scatter(reference[:, 0], reference[:, 1], s=8, c="tab:blue", label="reference")
    ax.scatter(query[:, 0], query[:, 1], s=8, c="tab:orange", label="new")
    for q, r in zip(query[:max_lines], reference[idx][:max_lines]):
        ax.plot([q[0], r[0]], [q[1], r[1]], "k-", lw=0.5, alpha=0.5)
    ax.set_title(f"{title}\nmean match distance = {dist.mean():.3f} m")
    ax.axis("equal"); ax.legend(loc="lower left")

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6))
draw_matches(ax_l, new, ref, "Guess: robot didn't move")
draw_matches(ax_r, to_world(new, pose_1), ref, "Guess: the true motion")
plt.show()

Two things stand out.

* With the *right* guess, almost every pairing joins two points on the same
  bit of wall and the match distances are tiny.
* With the *wrong* guess many pairings are wrong — but they are wrong in a
  useful way: most lines point roughly in the direction the scan needs to
  move. That is what makes ICP work: use the (imperfect) pairs to compute a
  better transform, re-pair, and repeat.

## Rejecting bad pairs

Some points in one scan have no true partner at all (parts of the room seen
from only one pose). A simple safeguard is to drop any pair whose distance
exceeds a threshold.

In [ ]:
dist, idx = KDTree(ref).query(new)
threshold = 0.5
keep = dist < threshold

print(f"{keep.sum()} of {len(new)} pairs kept (distance < {threshold} m)")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(ref[:, 0], ref[:, 1], s=8, c="tab:blue", label="reference")
ax.scatter(new[keep, 0], new[keep, 1], s=8, c="tab:green", label="new: kept")
ax.scatter(new[~keep, 0], new[~keep, 1], s=14, c="tab:red", marker="x", label="new: rejected")
ax.axis("equal"); ax.legend(); ax.grid(True)
plt.show()

## Summary

* Real scans have no built-in point pairing; nearest neighbour is the standard way to guess one.
* Brute force is a one-liner but scales as $N \times M$; a KD-tree scales as $N \log M$.
* `scipy.spatial.KDTree(reference).query(points)` returns `(distances, indices)`.
* Pairings are only as good as the current alignment guess — which is why ICP iterates.

**Next:** the SVD solver script (`4b`) computes the best rigid transform
for a *given* set of pairs, and `4c` closes the loop into full ICP.